# ML Assignment 2 – Bank Marketing Dataset

In [ ]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix, classification_report
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns

import joblib, os, warnings
warnings.filterwarnings("ignore")


In [ ]:

DATASET_PATH = "bank-additional-full.csv"
df = pd.read_csv(DATASET_PATH, sep=";")
df.head()


In [ ]:

print("Shape:", df.shape)
print("Target distribution:\n", df['y'].value_counts())
print("Missing values:", df.isnull().sum().sum())


In [ ]:

df['y'] = df['y'].map({'yes': 1, 'no': 0})


In [ ]:

categorical_cols = df.select_dtypes(include='object').columns
encoder = LabelEncoder()
for col in categorical_cols:
    df[col] = encoder.fit_transform(df[col])


In [ ]:

X = df.drop('y', axis=1)
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)


In [ ]:

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:

def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred)
    }


In [ ]:

results = {}

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": xgb.XGBClassifier(eval_metric='logloss', use_label_encoder=False, random_state=42)
}

for name, model in models.items():
    if name in ["Logistic Regression", "KNN"]:
        model.fit(X_train_scaled, y_train)
        results[name] = evaluate_model(model, X_test_scaled, y_test)
    else:
        model.fit(X_train, y_train)
        results[name] = evaluate_model(model, X_test, y_test)


In [ ]:

results_df = pd.DataFrame(results).T
results_df


In [ ]:

best_model = models["XGBoost"]
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - XGBoost")
plt.show()


In [ ]:

print(classification_report(y_test, y_pred))


In [ ]:

os.makedirs("model", exist_ok=True)

joblib.dump(models["Logistic Regression"], "model/logistic.pkl")
joblib.dump(models["Decision Tree"], "model/decision_tree.pkl")
joblib.dump(models["KNN"], "model/knn.pkl")
joblib.dump(models["Naive Bayes"], "model/naive_bayes.pkl")
joblib.dump(models["Random Forest"], "model/random_forest.pkl")
joblib.dump(best_model, "model/xgboost.pkl")
joblib.dump(scaler, "model/scaler.pkl")
